# 02 · Tasks and prompts

The model was trained to read a **short instruction in front of your text**, such as
`task: search result | query: `. The instruction tells it what the vector is for.
If you leave it out, you get noticeably worse vectors.

You never write these instructions yourself. You pick a **task**, and the engine writes them.
`eg.preview` shows the exact text the model receives, without loading anything.

In [1]:
import os
from pathlib import Path

import embeddinggemma_engine as eg

if Path.cwd().name == "notebooks":
    os.chdir("..")  # run from the repo root, so paths stay short
IMAGES = Path("assets/images")
VIDEOS = Path("assets/videos")
AUDIO = Path("assets/audio")


def mistake(fn):
    """Run fn and print the error the engine raises for it."""
    try:
        fn()
    except eg.EngineError as error:
        print(f"{type(error).__name__}: {error}\n")

## The seven tasks

| Task | Use it when | Kind |
|---|---|---|
| `search` | finding documents/media for a query | query ↔ document |
| `question_answering` | finding the passage that answers a question | query ↔ document |
| `fact_checking` | finding evidence for or against a claim | query ↔ document |
| `code_retrieval` | finding code from a description | query ↔ document |
| `similarity` | "how alike are these?" (duplicates, labelling) | everything the same |
| `clustering` | grouping items | everything the same |
| `classification` | features for a classifier you train | everything the same |

## Query and document get different prompts
In search, a query ("coffee?") and a document (a whole recipe) are different kinds of text.
The model is told which is which.

In [2]:
print(eg.preview("how do I make coffee?", task="search", side="query"))
print(eg.preview("Bloom the grounds, then pour slowly.", task="search", side="document"))

task: search result | query: how do I make coffee?
title: none | text: Bloom the grounds, then pour slowly.


Each query-style task has its own instruction. Documents always look the same.

In [3]:
for task in ["search", "question_answering", "fact_checking", "code_retrieval"]:
    print(eg.preview("my query", task=task, side="query"))

task: search result | query: my query
task: question answering | query: my query
task: fact checking | query: my query
task: code retrieval | query: my query


That is why there are two methods:

- `engine.embed_queries(...)` for the query side
- `engine.embed_documents(...)` for the document side

For the three "everything the same" tasks, both sides get the same prompt, and you use `engine.embed(...)`.

In [4]:
print(eg.preview("a photo of a cat", task="similarity"))
print(eg.preview("a photo of a cat", task="clustering"))

task: sentence similarity | query: a photo of a cat
task: clustering | query: a photo of a cat


## Titles
A document can carry a title. It goes into the prompt; without one, the model sees `title: none`.

In [5]:
print(eg.preview(eg.Item.text("Bloom the grounds first.", title="Pour-over coffee"), task="search", side="document"))
print(eg.preview(eg.Item.text("Bloom the grounds first."), task="search", side="document"))

title: Pour-over coffee | text: Bloom the grounds first.
title: none | text: Bloom the grounds first.


## Images, video and audio
A media file shows up as a placeholder (`<|image|>`, `<|video|>`, `<|audio|>`). The engine
replaces it with the media's own tokens.

**A media-only item gets no instruction.** The model card recommends this, and it measured best (see `research/`).
If you give media a title, the document prompt is added around it.

In [6]:
print(eg.preview(eg.Item.image(IMAGES / "kitchen.jpg"), task="search", side="document"))
print(eg.preview(eg.Item.image(IMAGES / "kitchen.jpg", title="Old oven"), task="search", side="document"))

<|image|>
title: Old oven | text: <|image|>


Text and media can be mixed in one item, in order. The instruction goes in front because the item has text.

In [7]:
listing = eg.Item.of("Coffee class.", eg.Video(VIDEOS / "04_coffee.mp4"), "Learn to use a gooseneck kettle.")
print(eg.preview(listing, task="search", side="document"))

title: none | text: Coffee class.<|video|>Learn to use a gooseneck kettle.


## Why it matters: the task is part of the vector
The same sentence embedded for different tasks gives different vectors.
So **queries and documents must use the same task**, and the engine checks this for you.

In [8]:
engine = eg.EmbeddingEngine()
sentence = ["Giraffes eat acacia leaves."]

as_search_doc = engine.embed_documents(sentence, task="search")
as_query = engine.embed_queries(sentence, task="search")
as_similarity = engine.embed(sentence, task="similarity")

print("search document vs search query:", round(float(eg.similarity(as_query, as_search_doc)[0, 0]), 3))
print("search document vs similarity:  ", round(float(as_search_doc.vectors[0] @ as_similarity.vectors[0]), 3))

search document vs search query: 0.968
search document vs similarity:   0.921


A query and a document with identical words score 0.97, not 1.0. That's expected:
the model knows one is a question and the other is an answer.

## Choosing a task
- Searching a collection (RAG, media search) → `search`. Switch to `question_answering`,
  `fact_checking` or `code_retrieval` when that is exactly what you are doing.
- Duplicates, "more like this", zero-shot labels → `similarity`.
- Grouping a dataset → `clustering`.

## Common mistakes

In [9]:
print("A query-style task through embed():")
mistake(lambda: engine.embed(["coffee"], task="search"))

print("A title on the query side (titles are for documents):")
mistake(lambda: engine.embed_queries([eg.Item.text("coffee", title="Coffee")], task="search"))

print("Comparing vectors made for different tasks:")
mistake(lambda: eg.similarity(as_query, as_similarity))

A query-style task through embed():
TaskError: search is asymmetric: use embed_queries / embed_documents, not embed.

A title on the query side (titles are for documents):
TaskError: item 0: title is only used by embed_documents (this is the query side).

Comparing vectors made for different tasks:
IncompatibleEmbeddingsError: Cannot compare these embeddings: task search vs similarity

